In [ ]:
import os
from openai import OpenAI

def build_repo_context(repo_path, ignore_dirs=None, extensions=None):
    """Walks a directory and bundles code files into a structured text string."""
    if ignore_dirs is None:
        ignore_dirs = {'.git', '__pycache__', 'node_modules', 'data', 'venv', '.venv'}
    if extensions is None:
        extensions = {'.py', '.ipynb', '.json', '.yaml', '.sh', '.md'} # ML relevant files
        
    context_pieces = []
    
    for root, dirs, files in os.walk(repo_path):
        # Skip hidden or ignored directories
        dirs[:] = [d for d in dirs if d not in ignore_dirs and not d.startswith('.')]
        
        for file in files:
            ext = os.path.splitext(file)[1]
            if ext in extensions:
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, repo_path)
                
                try:
                    with open(full_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                    
                    # Wrap file path and contents in clean markdown boundaries
                    context_pieces.append(f"\n[File: {rel_path}]\n```python\n{content}\n```\n")
                except Exception as e:
                    print(f"Skipping {rel_path} due to error: {e}")
                    
    return "\n".join(context_pieces)

# 1. Path to your local Git repository
REPO_PATH = "/workspaces/Research-Project-1" 

# 2. Bundle the files into a single text block
repo_string = build_repo_context(REPO_PATH)

# 3. Formulate your final instructions
user_instruction = "Analyze my training loops across these files. Why is my loss oscillating?"
final_prompt = f"{user_instruction}\n\n--- REPOSITORY FILES ---\n{repo_string}"

# # 4. Send directly to Ox Alpha via OpenRouter
# client = OpenAI(
#     base_url="https://openrouter.ai",
#     
# )

response = client.chat.completions.create(
    model="stealth/ox-alpha",
    messages=[{"role": "user", "content": final_prompt}]
)

print(response.choices[0].message.content)
